# 10 - PySpark Headroom and Risk Bands


## Setup
Target: Configure Spark + JDBC helper.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
spark = (SparkSession.builder.appName('capacity-demo').config('spark.driver.host','127.0.0.1').config('spark.driver.bindAddress','127.0.0.1').config('spark.jars.packages','org.postgresql:postgresql:42.7.4').getOrCreate())
JDBC_URL = 'jdbc:postgresql://host.docker.internal:5432/observability'
def load_query(q: str):
    return (spark.read.format('jdbc').option('url', JDBC_URL).option('dbtable', f'({q}) t').option('user','obs_user').option('password','obs_pass').option('driver','org.postgresql.Driver').load())


## Risk Bands
Target: Derive headroom, breach flag, and risk band.


In [ ]:
q = """SELECT sampled_at, host, cpu_pct, region AS application, env AS service FROM lab.telemetry_cpu_raw WHERE sampled_at >= now() - interval '14 days'"""
df = load_query(q)
h = df.withColumn('hour_bucket', F.date_trunc('hour', 'sampled_at')).groupBy('host','application','service','hour_bucket').agg(F.max('cpu_pct').alias('peak_cpu'))
w = Window.partitionBy('host','application','service').orderBy('hour_bucket').rowsBetween(-23, 0)
out = h.withColumn('cpu_24h_rolling_peak', F.max('peak_cpu').over(w)).withColumn('cpu_headroom_pct', F.lit(85) - F.col('cpu_24h_rolling_peak')).withColumn('cpu_breach_flag', F.col('cpu_24h_rolling_peak') >= 85).withColumn('cpu_risk_band', F.when(F.col('cpu_24h_rolling_peak') >= 85, 'breached').when((F.lit(85)-F.col('cpu_24h_rolling_peak')) <= 5, 'critical').when((F.lit(85)-F.col('cpu_24h_rolling_peak')) <= 15, 'warning').otherwise('healthy'))
out.orderBy(F.col('hour_bucket').desc(), 'host').show(40, truncate=False)
